# Rancher Kubernetes (RKE2) on FABRIC via Ansible

This notebook provisions a **2-node** FABRIC slice on a single site and uses Ansible to deploy [RKE2](https://docs.rke2.io/) (Rancher Kubernetes Engine 2).

| Role | Node | Resources |
|------|------|-----------|
| Control plane (`rke2-server`) | `node1` | 8 cores, 16 GB RAM |
| Worker (`rke2-agent`) | `node2` | 8 cores, 16 GB RAM |

Cluster traffic uses a private L2 dataplane (`192.168.1.0/24`). Ansible reaches the nodes over the FABRIC management network.

## Import the FABlib Library

In [1]:
from ipaddress import IPv4Network
import random
from pathlib import Path

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.verify_and_configure()

User: lngo@wcupa.edu bastion key is valid!
Configuration is valid
User: lngo@wcupa.edu bastion key is valid!
Configuration is valid
Please save the config!


## Find a Site With Enough Capacity

We need **2 nodes × 8 cores × 10 GB RAM** on one site. Requirements are scaled by `1.2` so a busy site is less likely to fail mid-submit.

In [14]:
resources = fablib.get_resources()
resources.update()

nodesReq = 2
coresReq = 8
ramReq = 10

# Scale up quite a bit to ensure there are plenty of resources available.
totalCoreAvail = nodesReq * coresReq * 5
totalRamAvail = nodesReq * ramReq * 5

usableSite = []
siteList = resources.get_site_names()
for site in siteList:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores >= totalCoreAvail and ram >= totalRamAvail:
        usableSite.append(site)

print(f"Need ~{totalCoreAvail:.0f} cores and ~{totalRamAvail:.0f} GB RAM available")
print(f"Usable sites ({len(usableSite)}): {usableSite}")
assert usableSite, "No site currently has enough free cores/RAM for this slice"

Need ~80 cores and ~100 GB RAM available
Usable sites (25): ['EDUKY', 'SEAT', 'PRIN', 'CERN', 'PSC', 'RUTG', 'AMST', 'FIU', 'MICH', 'SALT', 'NEWY', 'MASS', 'TOKY', 'TACC', 'WASH', 'DALL', 'MAX', 'NCSA', 'HAWI', 'GATECH', 'CLEM', 'UTAH', 'SRI', 'GPN', 'CIEN']


## Create the Slice

Site selection is random among usable sites so concurrent class users are less likely to collide.

In [15]:
siteName = random.choice(usableSite)
sliceName = "RancherK8s"
network_name = "rke2net"
print(f"Selected site: {siteName}")

slice = fablib.new_slice(name=sliceName)
net = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

for i in range(1, nodesReq + 1):
    node = slice.add_node(
        name=f"node{i}",
        site=siteName,
        cores=coresReq,
        ram=ramReq,
        disk=50,
        image="default_ubuntu_22",
    )
    iface = node.add_component(model="NIC_Basic", name="nic").get_interfaces()[0]
    iface.set_mode("config")
    net.add_interface(iface)

slice.submit()


Retry: 11, Time: 246 sec


ID,ed2da87c-d1cd-47b6-b275-2c80ab724ccb
Name,RancherK8s
Lease Expiration (UTC),2026-09-03 17:45:59 +0000
Lease Start (UTC),2026-09-02 17:45:59 +0000
Project ID,8b6dfc51-02ad-4f00-a389-6e75d8b61a26
State,StableOK
Email,lngo@wcupa.edu
UserId,8eecd713-fa8f-4b3b-8883-1ff9b021fa53


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
e02af539-881f-4476-b894-e5abf9a76106,node1,8,16,100,default_ubuntu_22,qcow2,salt-w3.fabric-testbed.net,SALT,ubuntu,2001:400:a100:3010:f816:3eff:fefc:a93c,Active,,ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3010:f816:3eff:fefc:a93c,/home/fabric/.ssh/slice_key.pub,/home/fabric/.ssh/slice_key
87619d90-0f0f-4f71-8086-75181fe5f2d9,node2,8,16,100,default_ubuntu_22,qcow2,salt-w3.fabric-testbed.net,SALT,ubuntu,2001:400:a100:3010:f816:3eff:fed8:4899,Active,,ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3010:f816:3eff:fed8:4899,/home/fabric/.ssh/slice_key.pub,/home/fabric/.ssh/slice_key


ID,Name,Layer,Type,Site,Gateway,Subnet,State,Error
340c7eaf-3db5-489d-bd7a-2693dad5b547,rke2net,L2,L2Bridge,SALT,None,192.168.1.0/24,Active,


Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
node1-nic-p1,p1,node1,rke2net,100,,06:A4:6F:48:CC:53,enp7s0,enp7s0,config,fe80::4a4:6fff:fe48:cc53,4,HundredGigE0/0/0/13
node2-nic-p1,p1,node2,rke2net,100,,0A:53:68:4D:9F:62,enp7s0,enp7s0,config,fe80::853:68ff:fe4d:9f62,4,HundredGigE0/0/0/13



Time to print interfaces 246 seconds


'ed2da87c-d1cd-47b6-b275-2c80ab724ccb'

In [16]:
import time

while True:
    time.sleep(10)
    slice.update()
    slice_state = slice.get_state()
    print(f"Slice state: {slice_state}")
    if slice_state == "Closing":
        print(f"Need to find new site")
        break
    else: 
        print("Slice stable:", slice.isStable())
    nodes = slice.get_nodes()
    if all(node.get_management_ip() is not None for node in nodes):
        for node in nodes:
            print("----", node.get_name(), "----")
            print("reservation state:", node.get_reservation_state())
            print("management ip:", node.get_management_ip())
            print("username:", node.get_username())
            print("error:", node.get_error_message())
            print(node.get_ssh_command())
        break

Slice state: StableOK
Slice stable: True
---- node1 ----
reservation state: Active
management ip: 2001:400:a100:3010:f816:3eff:fefc:a93c
username: ubuntu
error: 
ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3010:f816:3eff:fefc:a93c
---- node2 ----
reservation state: Active
management ip: 2001:400:a100:3010:f816:3eff:fed8:4899
username: ubuntu
error: 
ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3010:f816:3eff:fed8:4899


## Configure Dataplane Networking

Assign private IPs on the L2 network. RKE2 will advertise and join using these addresses.

In [17]:
for i in range(1, nodesReq + 1):
    node = slice.get_node(name=f"node{i}")
    iface = node.get_interface(network_name=network_name)
    iface.ip_link_up()
    iface.ip_addr_add(
        addr=f"192.168.1.{i}",
        subnet=IPv4Network("192.168.1.0/24"),
    )
    print(f"{node.get_name()} -> 192.168.1.{i}")

node1 -> 192.168.1.1
node2 -> 192.168.1.2


**Keep re-running the cell below until every node can ping every other node.**

In [18]:
for i in range(1, nodesReq):
    src = slice.get_node(name=f"node{i}")
    for j in range(i + 1, nodesReq + 1):
        des = slice.get_node(name=f"node{j}")
        des_addr = des.get_interface(network_name=network_name).get_ip_addr()
        print(f"{src.get_name()} is pinging {des.get_name()} at {des_addr} ========")
        stdout, stderr = src.execute(f"ping -c 2 {des_addr}")

node1 is pinging node2 at 192.168.1.2 ========
PING 192.168.1.2 (192.168.1.2) 56(84) bytes of data.
64 bytes from 192.168.1.2: icmp_seq=1 ttl=64 time=0.304 ms
64 bytes from 192.168.1.2: icmp_seq=2 ttl=64 time=0.061 ms

--- 192.168.1.2 ping statistics ---
2 packets transmitted, 2 received, 0% packet loss, time 1021ms
rtt min/avg/max/mdev = 0.061/0.182/0.304/0.121 ms


## Create Ansible Inventory

- `node1` → `rke2_servers` (control plane)
- `node2` → `rke2_agents` (worker)

A shared `rke2_token` is written into inventory so agents can join without scraping the server token file by hand.

In [19]:
node_defs = []
for node in slice.get_nodes():
    name = node.get_name()
    private_ip = str(node.get_interface(network_name=network_name).get_ip_addr())
    group = "rke2_servers" if name == "node1" else "rke2_agents"
    node_defs.append({"name": name, "private_ip": private_ip, "group": group})

slice_key = fablib.get_default_slice_key()["slice_private_key_file"]
ssh_config = "/home/fabric/work/fabric_config/ssh_config"

for nd in node_defs:
    node = slice.get_node(nd["name"])
    stdout, stderr = node.execute(
        "python3 -c 'import sys; print(sys.executable)'",
        quiet=True,
    )
    nd["node"] = node
    nd["python"] = stdout.strip()

lines = []
lines.append("all:")
lines.append("  vars:")
lines.append("    ansible_become: true")
lines.append(f'    ansible_ssh_private_key_file: "{slice_key}"')
lines.append(f'    ansible_ssh_common_args: "-F {ssh_config}"')
lines.append('    rke2_token: "fabric-rke2-cluster-token"')
lines.append('    rke2_server_private_ip: "192.168.1.1"')
lines.append("")
lines.append("  children:")

for group in ("rke2_servers", "rke2_agents"):
    lines.append(f"    {group}:")
    lines.append("      hosts:")
    for nd in node_defs:
        if nd["group"] != group:
            continue
        node = nd["node"]
        lines.append(f'        {nd["name"]}:')
        lines.append(f'          ansible_host: "{node.get_management_ip()}"')
        lines.append(f'          ansible_user: "{node.get_username()}"')
        lines.append(f'          ansible_python_interpreter: "{nd["python"]}"')
        lines.append(f'          private_ip: "{nd["private_ip"]}"')
    lines.append("")

inventory = "\n".join(lines) + "\n"
Path("playbook").mkdir(exist_ok=True)
Path("playbook/inventory.yml").write_text(inventory)

print("Wrote playbook/inventory.yml")
print(inventory)

Wrote playbook/inventory.yml
all:
  vars:
    ansible_become: true
    ansible_ssh_private_key_file: "/home/fabric/.ssh/slice_key"
    ansible_ssh_common_args: "-F /home/fabric/work/fabric_config/ssh_config"
    rke2_token: "fabric-rke2-cluster-token"
    rke2_server_private_ip: "192.168.1.1"

  children:
    rke2_servers:
      hosts:
        node1:
          ansible_host: "2001:400:a100:3010:f816:3eff:fefc:a93c"
          ansible_user: "ubuntu"
          ansible_python_interpreter: "/usr/bin/python3"
          private_ip: "192.168.1.1"

    rke2_agents:
      hosts:
        node2:
          ansible_host: "2001:400:a100:3010:f816:3eff:fed8:4899"
          ansible_user: "ubuntu"
          ansible_python_interpreter: "/usr/bin/python3"
          private_ip: "192.168.1.2"




## Playbook: Host Prerequisites

Disables swap, loads `overlay` / `br_netfilter`, enables IP forwarding, and writes `/etc/hosts` entries for the cluster nodes.

In [20]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-prereqs.yml


PLAY [Prepare hosts for RKE2] **************************************************

TASK [Gathering Facts] *********************************************************
ok: [node2]
ok: [node1]

TASK [Install prerequisite packages] *******************************************
changed: [node1]
changed: [node2]

TASK [Disable swap (required by Kubernetes)] ***********************************
ok: [node2]
ok: [node1]

TASK [Ensure swap stays disabled across reboots] *******************************
ok: [node1]
ok: [node2]

TASK [Load overlay kernel module] **********************************************
ok: [node1]
ok: [node2]

TASK [Load br_netfilter kernel module] *****************************************
ok: [node1]
ok: [node2]

TASK [Persist kernel module loads] *********************************************
changed: [node1]
changed: [node2]

TASK [Configure sysctl for Kubernetes networking] ******************************
changed: [node2]
changed: [node1]

TASK [Apply sysctl settings] **********

## Playbook: Install RKE2

Declarative take on the [RKE2 quick start](https://docs.rke2.io/install/quickstart):

1. Write `config.yaml` so the server advertises on the dataplane IP
2. Install and start `rke2-server` on `node1`
3. Install and start `rke2-agent` on `node2` using the shared token
4. Deploy a small `nginx` Deployment + NodePort Service (`30080`) for a smoke test

First-time install commonly takes **10–15 minutes** while images are pulled.

In [21]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-rke2.yml


PLAY [Bootstrap RKE2 server (control plane)] ***********************************

TASK [Gathering Facts] *********************************************************
ok: [node1]

TASK [Ensure RKE2 config directory exists] *************************************
changed: [node1]

TASK [Discover dataplane interface for private IP] *****************************
ok: [node1]

TASK [Show dataplane interface] ************************************************
ok: [node1] => {
    "msg": "Using dataplane iface enp7s0 for 192.168.1.1"
}

TASK [Write RKE2 server config (advertise on private dataplane IP)] ************
changed: [node1]

TASK [Install RKE2 server] *****************************************************
changed: [node1]

TASK [Ensure server manifests directory exists] ********************************
changed: [node1]

TASK [Write Canal HelmChartConfig for FABRIC dataplane] ************************
changed: [node1]

TASK [Enable and start rke2-server] ***************************************

## Verify the Cluster

In [22]:
server = slice.get_node("node1")

print("==== nodes ====")
stdout, stderr = server.execute("kubectl get nodes -o wide", quiet=True)
print(stdout)

print("==== nginx-demo ====")
stdout, stderr = server.execute("kubectl get pods,svc -l app=nginx-demo -o wide", quiet=True)
print(stdout)

print("==== curl via NodePort on dataplane ====")
stdout, stderr = server.execute("curl -s -o /dev/null -w '%{http_code}\n' http://192.168.1.1:30080/", quiet=True)
print("HTTP status:", stdout.strip())

==== nodes ====
NAME    STATUS   ROLES                AGE    VERSION          INTERNAL-IP   EXTERNAL-IP   OS-IMAGE             KERNEL-VERSION               CONTAINER-RUNTIME
node1   Ready    control-plane,etcd   110s   v1.36.4+rke2r1   192.168.1.1   <none>        Ubuntu 22.04.5 LTS   5.15.0-185-generic (amd64)   containerd://2.3.4-k3s1.36
node2   Ready    <none>               46s    v1.36.4+rke2r1   192.168.1.2   <none>        Ubuntu 22.04.5 LTS   5.15.0-185-generic (amd64)   containerd://2.3.4-k3s1.36

==== nginx-demo ====
NAME                             READY   STATUS    RESTARTS   AGE   IP           NODE    NOMINATED NODE   READINESS GATES
pod/nginx-demo-6f8d7bb5d-f8x59   1/1     Running   0          7s    10.42.1.2    node2   <none>           <none>
pod/nginx-demo-6f8d7bb5d-flh6h   1/1     Running   0          7s    10.42.0.10   node1   <none>           <none>

==== curl via NodePort on dataplane ====
HTTP status: 000


## Useful Follow-ups

On `node1` (after SSH):

```bash
kubectl get pods -A
kubectl describe node node1
sudo journalctl -u rke2-server -f
```

On `node2`:

```bash
sudo journalctl -u rke2-agent -f
```

To reach the NodePort from your laptop, create an SSH tunnel to `node1:30080` (same pattern as the Docker Swarm notebook).

## Start the SSH Tunnel

- Create SSH Tunnel Configuration `fabric_ssh_tunnel_tools.zip`
- Download your custom `fabric_ssh_tunnel_tools.zip` tarball from the `fabric_config` folder.  
- Untar the tarball and put the resulting folder (`fabric_ssh_tunnel_tools`) somewhere you can access it from the command line.
- Open a terminal window. (Windows: use `powershell`) 
- Use `cd` to navigate to the `fabric_ssh_tunnel_tools` folder.
- Run the following command to setup permission correctly

```bash
chmod 600 slice_key fabric-bastion-key
```

- In your terminal, run the command that results from running the following cell (leave the terminal window open).

In [23]:
fablib.create_ssh_tunnel_config(overwrite=True)


SSH tunnel config created and zipped at: /home/fabric/work/fabric_config/fabric_ssh_tunnel_tools.tgz

Download Instructions:
Download your custom `fabric_ssh_tunnel_tools.tgz` file from the `fabric_config` folder.

Usage Instructions:
1. Unzip the archive and place the resulting `fabric_ssh_tunnel_tools/` folder somewhere accessible from your terminal.
2. Open a terminal window (on Windows, use PowerShell).
3. Use `cd` to navigate into the `fabric_ssh_tunnel_tools` folder.
4. In your terminal, run the SSH tunnel command generated by the next notebook cell.
    


In [32]:
import os
# Port on your local machine that you want to map the web server to. This should be a port that you have specified on 
# docker-compose.yml

server = slice.get_node("node1")

print(f"Open a terminal tab here and run the following command to connect to the RKE2 server: \n")
print(f'ssh  -i {os.path.basename(fablib.get_default_slice_public_key_file())[:-4]} -F ssh_config {target_host}')
print(f"\nRun the following command once you are connected: \n")
print("kubectl port-forward --address 127.0.0.1 svc/nginx-demo 8080:80")

Open a terminal tab here and run the following command to connect to the RKE2 server: 

ssh  -i slice_key -F ssh_config ubuntu@2001:400:a100:3010:f816:3eff:fefc:a93c

Run the following command once you are connected: 

kubectl port-forward --address 127.0.0.1 svc/nginx-demo 8080:80


In [33]:
import os
# Port on your local machine that you want to map the web server to. This should be a port that you have specified on 
# docker-compose.yml

server = slice.get_node("node1")
local_port='5555'
# We use 0.0.0.0 because we want the ability to forward this interface outside of the container. 
local_host='0.0.0.0'

# Port on the node used by the web server
target_port='8080'

# Username/node on FABRIC
target_host=f'{server.get_username()}@{server.get_management_ip()}'
print(f"Open another terminal tab here and run the following command to open the SSH tunnel: \n")
print(f'ssh  -L {local_host}:{local_port}:127.0.0.1:{target_port} -i {os.path.basename(fablib.get_default_slice_public_key_file())[:-4]} -F ssh_config {target_host}')

Open another terminal tab here and run the following command to open the SSH tunnel: 

ssh  -L 0.0.0.0:5555:127.0.0.1:8080 -i slice_key -F ssh_config ubuntu@2001:400:a100:3010:f816:3eff:fefc:a93c


Open a new browser tab and visit `127.0.0.1:5555`

## Cleanup

In [34]:
# Uncomment when you are finished
slice.delete()